### np.dot

In [4]:
def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def _product(shape):
    p = 1
    for s in shape:
        p *= s
    return p


def _get_item(a, idx):
    cur = a
    for i in idx:
        cur = cur[i]
    return cur


def _build_nested(shape, getter, prefix=()):
    if len(shape) == 0:
        return getter(prefix)
    return [_build_nested(shape[1:], getter, prefix + (i,)) for i in range(shape[0])]


def np_dot(a, b):
    shape_a = get_shape(a)
    shape_b = get_shape(b)
    ndim_a = len(shape_a)
    ndim_b = len(shape_b)

    if ndim_a == 0 and ndim_b == 0:
        return a * b

    if ndim_a == 0 or ndim_b == 0:
        raise ValueError("dot does not support scalar with non-scalar in this implementation")

    if ndim_a == 1 and ndim_b == 1:
        if shape_a[0] != shape_b[0]:
            raise ValueError("shapes not aligned for 1D dot")
        total = 0
        for i in range(shape_a[0]):
            total += a[i] * b[i]
        return total

    if ndim_a >= 2 and ndim_b == 1:
        if shape_a[-1] != shape_b[0]:
            raise ValueError("shapes not aligned for N-D x 1-D dot")

        out_shape = shape_a[:-1]

        if len(out_shape) == 0:
            total = 0
            for k in range(shape_b[0]):
                total += a[k] * b[k]
            return total

        def getter(out_idx):
            total = 0
            for k in range(shape_b[0]):
                idx_a = out_idx + (k,)
                total += _get_item(a, idx_a) * b[k]
            return total

        return _build_nested(out_shape, getter)

    if ndim_a == 1 and ndim_b >= 2:
        if shape_a[0] != shape_b[-2]:
            raise ValueError("shapes not aligned for 1-D x N-D dot")

        out_shape = shape_b[:-2] + (shape_b[-1],)

        if len(out_shape) == 0:
            total = 0
            for k in range(shape_a[0]):
                total += a[k] * _get_item(b, (k,))
            return total

        def getter(out_idx):
            if len(shape_b) == 2:
                prefix = ()
                j = out_idx[0]
            else:
                prefix = out_idx[:-1]
                j = out_idx[-1]

            total = 0
            for k in range(shape_a[0]):
                idx_b = prefix + (k, j)
                total += a[k] * _get_item(b, idx_b)
            return total

        return _build_nested(out_shape, getter)

    if ndim_a >= 2 and ndim_b >= 2:
        if shape_a[-1] != shape_b[-2]:
            raise ValueError("incompatible shapes: last axis of a must match second-to-last axis of b")

        out_shape = shape_a[:-1] + shape_b[:-2] + (shape_b[-1],)

        def getter(out_idx):
            left_len = ndim_a - 1
            right_prefix_len = ndim_b - 2

            idx_a_prefix = out_idx[:left_len]
            idx_b_prefix = out_idx[left_len:left_len + right_prefix_len]
            j = out_idx[-1]

            total = 0
            for k in range(shape_a[-1]):
                idx_a = idx_a_prefix + (k,)
                idx_b = idx_b_prefix + (k, j)
                total += _get_item(a, idx_a) * _get_item(b, idx_b)
            return total

        return _build_nested(out_shape, getter)

    raise ValueError("unsupported input shapes")

### test cases 

In [5]:
print(np_dot(2, 3))
print(np_dot([1, 2, 3], [4, 5, 6]))
print(np_dot([[1, 2], [3, 4]], [[5, 6], [7, 8]]))
print(np_dot([1, 2], [3, 4]))
print(np_dot([[1, 2, 3]], [[4], [5], [6]]))
print(np_dot([[[1, 2]]], [[[3], [4]]]))
print(np_dot([[1, 2, 3], [4, 5, 6]], [7, 8, 9]))

6
32
[[19, 22], [43, 50]]
11
[[32]]
[[[[11]]]]
[50, 122]


In [6]:
print(np_dot([], [1, 2, 3]))

ValueError: shapes not aligned for 1D dot

In [7]:
print(np_dot([1, 2], [1, 2, 3]))

ValueError: shapes not aligned for 1D dot

In [8]:
print(np_dot([[1, 2], [3, 4]], [[5, 6, 7]]))

ValueError: incompatible shapes: last axis of a must match second-to-last axis of b

In [9]:
print(np_dot([1, 2, 3], 4))

ValueError: dot does not support scalar with non-scalar in this implementation

In [10]:
print(np_dot([[1, 2], [3]] , [[1], [2]]))

ValueError: Jagged array: first row shape (2,), but row 1 shape (1,)